# Convolution Neural Network (CNN)

A CNN is a deep learning model which is designed for processing grid-like data such as images. In this case, the model will be used to detect whether the MRI image contains a brain tumor, and classify them accordingly.

## Setup

In [6]:
from pathlib import Path
import copy
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

### GPU Setup (NVIDIA CUDA)

This notebook uses **PyTorch** to train a convolutional neural network (CNN). When a CUDA-enabled PyTorch build and a compatible NVIDIA driver are available, training runs on the NVIDIA GPU; otherwise, it falls back to the CPU.

The following pip command was used to install PyTorch in a Jupyter code block:
- ```%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132```

PyTorch was installed in the notebook's Python environment with CUDA support. The CUDA-enabled PyTorch package provides the CUDA runtime it needs. Installing the full CUDA Toolkit separately is generally not required for this notebook.

However, the full toolkit can be accessed here:
- https://developer.nvidia.com/cuda-downloads?target_os=Windows&target_arch=x86_64&target_version=11&target_type=exe_local

Check that PyTorch can access the GPU:

```python
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Load data and prepare batches

In [7]:
DATA_DIR = Path("../data/processed")

def load_split(name):
    # These project NPZ files were generated locally and store y as object labels.
    with np.load(DATA_DIR / f"{name}.npz", allow_pickle=True) as data:
        images = data["X"]
        labels = data["y"].astype(str)
    return images, labels

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

classes = sorted(set(y_train))
label_to_id = {label: i for i, label in enumerate(classes)}

def encode_labels(labels):
    return np.array([label_to_id[label] for label in labels], dtype=np.int64)

class MRIDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images  # Keep uint8 in memory; convert per image in __getitem__.
        self.labels = encode_labels(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image = self.images[index].astype(np.float32)
        image_min, image_max = image.min(), image.max()
        image = (image - image_min) / (image_max - image_min + 1e-8)
        image = torch.from_numpy(image).unsqueeze(0)  # (1, 224, 224)
        label = torch.tensor(self.labels[index], dtype=torch.long)
        return image, label

train_loader = DataLoader(
    MRIDataset(X_train, y_train), batch_size=32, shuffle=True
)
val_loader = DataLoader(
    MRIDataset(X_val, y_val), batch_size=32, shuffle=False
)
test_loader = DataLoader(
    MRIDataset(X_test, y_test), batch_size=32, shuffle=False
)

print("Classes:", classes)
print("Training images:", len(X_train))

Classes: [np.str_('glioma'), np.str_('meningioma'), np.str_('notumor'), np.str_('pituitary')]
Training images: 4728


## Define CNN

In [8]:
class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainTumorCNN(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("Using:", device)

Using: cuda


## Train and keep best validation checkpoint

Ideally, across epochs:

- Training loss generally decreases, and training accuracy generally increases.
- Validation loss decreases too, while validation accuracy improves. These can fluctuate a little from epoch to epoch.
- Training and validation results stay reasonably close. If training keeps improving while validation loss rises or validation accuracy stalls, the model may be overfitting.

In [9]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)

    return total_loss / count, correct / count

max_epochs = 50
early_stopping_patience = 12
min_delta = 1e-4

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=min_delta,
    threshold_mode="abs",
    min_lr=1e-6,
)

best_val_loss = float("inf")
epochs_without_improvement = 0
best_state = None

for epoch in range(1, max_epochs + 1):
    model.train()
    train_loss_total = 0.0
    train_correct = 0
    train_count = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss_total += loss.item() * labels.size(0)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_count += labels.size(0)

    train_loss = train_loss_total / train_count
    train_accuracy = train_correct / train_count
    val_loss, val_accuracy = evaluate(model, val_loader)

    if val_loss < best_val_loss - min_delta:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
    else:
        epochs_without_improvement += 1

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:02d}/{max_epochs} | "
        f"train loss {train_loss:.4f}, accuracy {train_accuracy:.3f} | "
        f"val loss {val_loss:.4f}, accuracy {val_accuracy:.3f} | "
        f"lr {current_lr:.2e}"
    )

    if epochs_without_improvement >= early_stopping_patience:
        print(f"Early stopping after epoch {epoch}.")
        break

assert best_state is not None, "Training did not produce a valid checkpoint."
model.load_state_dict(best_state)

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)
torch.save(
    {"model_state_dict": model.state_dict(), "classes": classes},
    model_dir / "cnn_best.pt",
)

print(f"Best validation loss: {best_val_loss:.4f}")

Epoch 01/50 | train loss 0.8108, accuracy 0.666 | val loss 0.5919, accuracy 0.732 | lr 1.00e-03
Epoch 02/50 | train loss 0.5901, accuracy 0.770 | val loss 0.4404, accuracy 0.845 | lr 1.00e-03
Epoch 03/50 | train loss 0.5170, accuracy 0.800 | val loss 0.4825, accuracy 0.823 | lr 1.00e-03
Epoch 04/50 | train loss 0.4497, accuracy 0.825 | val loss 0.4080, accuracy 0.843 | lr 1.00e-03
Epoch 05/50 | train loss 0.4285, accuracy 0.827 | val loss 1.0640, accuracy 0.579 | lr 1.00e-03
Epoch 06/50 | train loss 0.3897, accuracy 0.850 | val loss 0.2730, accuracy 0.909 | lr 1.00e-03
Epoch 07/50 | train loss 0.3698, accuracy 0.858 | val loss 0.8106, accuracy 0.679 | lr 1.00e-03
Epoch 08/50 | train loss 0.3619, accuracy 0.862 | val loss 0.3392, accuracy 0.874 | lr 1.00e-03
Epoch 09/50 | train loss 0.3377, accuracy 0.872 | val loss 0.5280, accuracy 0.773 | lr 5.00e-04
Epoch 10/50 | train loss 0.2949, accuracy 0.887 | val loss 0.2477, accuracy 0.914 | lr 5.00e-04
Epoch 11/50 | train loss 0.2770, accurac

## Evaluate once on held-out test split

In [10]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_pred.extend(logits.argmax(dim=1).cpu().numpy())
        y_true.extend(labels.numpy())

print(classification_report(
    y_true,
    y_pred,
    labels=range(len(classes)),
    target_names=classes,
    zero_division=0,
))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

      glioma       0.98      0.69      0.81       386
  meningioma       0.80      0.83      0.82       398
     notumor       0.81      1.00      0.89       400
   pituitary       0.97      0.97      0.97       400

    accuracy                           0.88      1584
   macro avg       0.89      0.87      0.87      1584
weighted avg       0.89      0.88      0.87      1584

Confusion matrix:
[[268  70  44   4]
 [  6 332  51   9]
 [  0   1 399   0]
 [  0  13   0 387]]
